In [6]:
from datasets import Dataset, DatasetDict, load_from_disk
from collections import Counter

In [7]:
def load(path: str = "../output", split: str = "train"):
    ds = load_from_disk(path)
    print()
    ds_split = ds[split]
    return ds, ds_split

def overview(ds: DatasetDict):
    for split, data in ds.items():
        print(f"Split {split}:")
        print(f"Rows: {len(data)}")
        print(f"Columns: {data.column_names}")
        print(f"Features: {data.features}")
        print()

def stats(ds: Dataset):
    difficulties = Counter(ds["difficulty"])
    print("Difficulty:")
    for k, v in sorted(difficulties.items()):
        pct = v / len(ds) * 100
        print(f"{k:<10} {v:>5}  ({pct:.1f}%)")
    print()

    sources = Counter(ds["source"])
    print("Source:")
    for k, v in sorted(sources.items()):
        pct = v / len(ds) * 100
        print(f"{k:<15} {v:>5}  ({pct:.1f}%)")
    print()

    all_tags = [tag for tags in ds["tags"] if tags for tag in tags]
    top_tags = Counter(all_tags).most_common(10)
    print("Most common tags:")
    for tag, count in top_tags:
        print(f"{tag:<30} {count}")
    print()

In [8]:
def check_nulls(ds: Dataset):
    print(f"Null check")
    fields = ["slug", "difficulty", "tags", "problem", "starter_code", "solution", "source"]
    for field in fields:
        empty = sum(
            1 for val in ds[field]
            if val is None or val == "" or val == []
        )
        status = "o" if empty == 0 else "x"
        print(f"{status} {field:<15} {empty} empty rows")
    print()

def diff(train: Dataset, test: Dataset):
    print("\nTrain and test overlap")
    train_ids = set(t.strip().lower() for t in train["slug"])
    test_ids  = set(t.strip().lower() for t in test["slug"])
    overlap   = train_ids & test_ids

    if not overlap:
        print("No overlap")
    else:
        print(f"  {len(overlap)} slugs appear in both splits:")
        for slug in sorted(overlap)[:10]:
            print(f"    {slug}")

In [9]:
def sample(ds: Dataset, n: int = 3, source: str = ""):
    if source:
        ds = ds.filter(lambda x: x["source"] == source)
    print(f"Sample rows (source={source or 'any'})")
    for i in range(min(n, len(ds))):
        row = ds[i]
        print(f"\n[{i}] slug: {row['slug']}")
        print(f"difficulty:   {row['difficulty']}")
        print(f"tags:         {row['tags']}")
        print(f"source:       {row['source']}")
        print(f"problem:      {row['problem'][:120].strip()}...")
        print(f"starter_code: {row['starter_code'][:80].strip()}...")
        print(f"solution:     {row['solution'][:80].strip()}...")

In [10]:
dataset, train = load(split="train")
_, test = load(split="test")

if isinstance(dataset, DatasetDict):
    overview(dataset)

if isinstance(train, Dataset):
    stats(train)
    check_nulls(train)
    sample(train)

if isinstance(train, Dataset) and isinstance(test, Dataset):
    diff(train, test)



Split train:
Rows: 5000
Columns: ['difficulty', 'tags', 'starter_code', 'slug', 'problem', 'solution', 'tests', 'source']
Features: {'difficulty': Value('string'), 'tags': List(Value('string')), 'starter_code': Value('string'), 'slug': Value('string'), 'problem': Value('string'), 'solution': Value('string'), 'tests': List({'input': Value('string'), 'output': Value('string')}), 'source': Value('string')}

Split test:
Rows: 228
Columns: ['difficulty', 'tags', 'starter_code', 'slug', 'problem', 'solution', 'tests', 'source']
Features: {'difficulty': Value('string'), 'tags': List(Value('string')), 'starter_code': Value('string'), 'slug': Value('string'), 'problem': Value('string'), 'solution': Value('string'), 'tests': List({'input': Value('string'), 'output': Value('string')}), 'source': Value('string')}

Difficulty:
Easy        1178  (23.6%)
Hard        1144  (22.9%)
Medium      2678  (53.6%)

Source:
greengerong      2359  (47.2%)
newfacade        2641  (52.8%)

Most common tags:
Arra